In [16]:
# =====================================================
# AI-Enhanced Departmental Store Management System
# Actors: Admin, Manager, Customer, Supplier
# Language: Python
# =====================================================

products = {
    1: {"name": "Sugar", "price": 120, "stock": 50, "sales": [5, 6, 7]},
    2: {"name": "Rice", "price": 250, "stock": 40, "sales": [6, 5, 8]},
    3: {"name": "Oil", "price": 550, "stock": 25, "sales": [3, 4, 5]},
    4: {"name": "Flour", "price": 160, "stock": 60, "sales": [7, 6, 8]},
    5: {"name": "Milk", "price": 90, "stock": 70, "sales": [10, 12, 8]},
    6: {"name": "Tea", "price": 200, "stock": 35, "sales": [4, 5, 6]},
    7: {"name": "Coffee", "price": 350, "stock": 20, "sales": [3, 2, 4]},
    8: {"name": "Bread", "price": 50, "stock": 80, "sales": [15, 18, 20]},
}

LOW_STOCK_LIMIT = 10
customer_cart = []        # Cart to store purchased items temporarily
customer_history = []     # Store past purchased products for AI recommendations

# -----------------------------------------------------
# COMMON FUNCTION: View Products
# -----------------------------------------------------
def view_products():
    print("\nID  Name       Price   Stock")
    print("-------------------------------")
    for pid, data in products.items():
        print(f"{pid:<3} {data['name']:<10} {data['price']:<7} {data['stock']}")

# -----------------------------------------------------
# ADMIN FUNCTIONS
# -----------------------------------------------------
def add_product():
    pid = int(input("Enter Product ID: "))
    name = input("Enter Product Name: ")
    price = float(input("Enter Price: "))
    stock = int(input("Enter Stock: "))
    products[pid] = {"name": name, "price": price, "stock": stock, "sales": []}
    print("✔ Product added successfully!")

def restock_product():
    pid = int(input("Enter Product ID: "))
    qty = int(input("Enter quantity to add: "))
    if pid in products:
        products[pid]["stock"] += qty
        print("✔ Stock updated!")
    else:
        print("❌ Product not found!")

def admin_menu():
    while True:
        print("\n------ ADMIN MENU ------")
        print("1. View Products")
        print("2. Add Product")
        print("3. Restock Product")
        print("4. Back")
        ch = input("Enter choice: ")
        if ch == "1":
            view_products()
        elif ch == "2":
            add_product()
        elif ch == "3":
            restock_product()
        elif ch == "4":
            break
        else:
            print("❌ Invalid choice")

# -----------------------------------------------------
# MANAGER FUNCTIONS (AI)
# -----------------------------------------------------
def low_stock_report():
    found = False
    for data in products.values():
        if data["stock"] <= LOW_STOCK_LIMIT:
            found = True
            break
    if found:
        print("\n⚠ Stock is low!")  # Generic message only
    else:
        print("✔ No low-stock items")

def sales_analysis():
    print("\n--- Sales Analysis ---")
    print("\nName       Total Sales")
    print("----------------------")
    for data in products.values():
        print(f"{data['name']:<10} {sum(data['sales'])}")

def demand_prediction():
    print("\n--- AI Demand Prediction ---")
    print("\nName       Avg Demand")
    print("----------------------")
    for data in products.values():
        if data["sales"]:
            avg = sum(data["sales"]) / len(data["sales"])
            print(f"{data['name']:<10} {round(avg)}")
        else:
            print(f"{data['name']:<10} 0")

def manager_menu():
    while True:
        print("\n------ MANAGER MENU ------")
        print("1. View Products")
        print("2. Low Stock Report")
        print("3. Sales Analysis")
        print("4. Demand Prediction")
        print("5. Back")
        ch = input("Enter choice: ")
        if ch == "1":
            view_products()
        elif ch == "2":
            low_stock_report()
        elif ch == "3":
            sales_analysis()
        elif ch == "4":
            demand_prediction()
        elif ch == "5":
            break
        else:
            print("❌ Invalid choice")

# -----------------------------------------------------
# CUSTOMER FUNCTIONS (AI Enhanced)
# -----------------------------------------------------
def recommend_products():
    """Recommend products based on past purchases or default popular items"""
    if not customer_history:
        # First-time customer: show top popular products
        print("\n--- Recommended for you ---")
        popular = sorted(products.items(), key=lambda x: sum(x[1]['sales']), reverse=True)[:3]
        for pid, data in popular:
            print(f"Try {data['name']}!")
        return

    # Returning customer: recommend based on last 3 purchases
    print("\n--- Recommended for you ---")
    recommended = set()
    for pid in customer_history[-3:]:  # Last 3 purchases
        if pid == 5:  # Milk -> suggest Sugar or Bread
            recommended.update([1, 8])
        elif pid == 1:  # Sugar -> suggest Milk or Bread
            recommended.update([5, 8])
        elif pid == 8:  # Bread -> suggest Milk
            recommended.update([5])
    # Exclude items already in current cart
    recommended = recommended - set([item['pid'] for item in customer_cart])

    if recommended:
        for pid in recommended:
            print(f"Try {products[pid]['name']}!")
    else:
        print("No new recommendations.")

def buy_product():
    while True:
        view_products()
        recommend_products()  # AI Suggestion
        pid = int(input("Enter Product ID to buy: "))
        qty = int(input("Enter Quantity: "))
        if pid in products and products[pid]["stock"] >= qty:
            products[pid]["stock"] -= qty
            products[pid]["sales"].append(qty)
            customer_cart.append({"pid": pid, "qty": qty})
            customer_history.append(pid)
            print(f"✔ {products[pid]['name']} added to cart!")
        else:
            print("❌ Invalid product or insufficient stock")

        # Ask if customer wants to continue shopping
        cont = input("\nDo you want to continue shopping? (Yes/No): ").strip().lower()
        if cont != "yes":  # Agar customer "No" bole
            print("\n*****************************")
            print("  THANKS FOR SHOPPING!  ")
            print("*****************************\n")
            return  # Direct main menu pe wapas

def view_bill():
    if not customer_cart:
        print("\n🛒 No items in cart!")
        return

    total = 0
    print("\n----- BILL SUMMARY -----")
    print("Name       Qty  Price  Total")
    print("----------------------------")
    for item in customer_cart:
        pid = item["pid"]
        qty = item["qty"]
        price = products[pid]["price"]
        item_total = price * qty
        total += item_total
        print(f"{products[pid]['name']:<10} {qty:<3} {price:<6} {item_total}")

    tax = round(total * 0.05)
    grand_total = total + tax
    print("----------------------------")
    print(f"Subtotal: {total}")
    print(f"Tax (5%): {tax}")
    print(f"Grand Total: {grand_total}")
    print("----------------------------")

def customer_menu():
    global customer_cart
    customer_cart = []  # Reset cart on menu entry
    while True:
        print("\n------ CUSTOMER MENU ------")
        print("1. View Products")
        print("2. Buy Product")
        print("3. View Bill")
        # Back option removed
        ch = input("Enter choice: ")
        if ch == "1":
            view_products()
        elif ch == "2":
            buy_product()  # Customer automatically main menu pe wapas jayega if No
            return
        elif ch == "3":
            view_bill()
        else:
            print("❌ Invalid choice")

# -----------------------------------------------------
# SUPPLIER FUNCTIONS (AI Enhanced)
# -----------------------------------------------------
def supply_products():
    print("\n--- AI Suggested Supply Quantity ---")
    for pid, data in products.items():
        if data["sales"]:
            predicted = round(sum(data["sales"])/len(data["sales"])) * 2  # simple prediction
            print(f"{data['name']}: Suggested Supply = {predicted} units")
    pid = int(input("\nEnter Product ID to supply: "))
    qty = int(input("Enter Supply Quantity: "))
    if pid in products:
        products[pid]["stock"] += qty
        print("✔ Supply delivered successfully!")
    else:
        print("❌ Product not found")

def supplier_menu():
    while True:
        print("\n------ SUPPLIER MENU ------")
        view_products()
        print("1. Supply Products")
        print("2. Back")
        ch = input("Enter choice: ")
        if ch == "1":
            supply_products()
        elif ch == "2":
            break
        else:
            print("❌ Invalid choice")

# -----------------------------------------------------
# MAIN SYSTEM
# -----------------------------------------------------
def main():
    while True:
        print("\n====== Departmental Store System ======")
        print("1. Admin")
        print("2. Manager")
        print("3. Supplier")
        print("4. Customer")
        print("5. Exit")
        choice = input("Select role: ")
        if choice == "1":
            admin_menu()
        elif choice == "2":
            manager_menu()
        elif choice == "3":
            supplier_menu()
        elif choice == "4":
            customer_menu()
        elif choice == "5":
            print("System closed.")
            break
        else:
            print("❌ Invalid option")

# -----------------------------------------------------
# PROGRAM START
# -----------------------------------------------------
main()



====== Departmental Store System ======
1. Admin
2. Manager
3. Supplier
4. Customer
5. Exit
Select role: 1

------ ADMIN MENU ------
1. View Products
2. Add Product
3. Restock Product
4. Back
Enter choice: 1

ID  Name       Price   Stock
-------------------------------
1   Sugar      120     50
2   Rice       250     40
3   Oil        550     25
4   Flour      160     60
5   Milk       90      70
6   Tea        200     35
7   Coffee     350     20
8   Bread      50      80

------ ADMIN MENU ------
1. View Products
2. Add Product
3. Restock Product
4. Back
Enter choice: 2
Enter Product ID: 9
Enter Product Name: Crackers
Enter Price: 500
Enter Stock: 50
✔ Product added successfully!

------ ADMIN MENU ------
1. View Products
2. Add Product
3. Restock Product
4. Back
Enter choice: 3
Enter Product ID: 3
Enter quantity to add: 25
✔ Stock updated!

------ ADMIN MENU ------
1. View Products
2. Add Product
3. Restock Product
4. Back
Enter choice: 4

====== Departmental Store System ======
1.